# MDI3003: Advanced Predictive Analytics
## Laboratory Manual – Lab 04: Probabilistic Customer Segmentation and Segment Prediction Using Demographic, Psychographic, and Behavioral Data with Naive Bayes Classifiers
**Author:** Madhusudhanan G  
**Registration Number:** 23MID0444  
**Faculty:** Dr. Durgesh Kumar  
**School:** School of Computer Science and Engineering (SCOPE), VIT Vellore  
**Date:** August 18, 2026  

---
### Technical Positioning & Governance Notice
> **Important:** Naive Bayes is a supervised classification method. Therefore, this laboratory predicts predefined customer-segment labels. It does not discover segments through unsupervised clustering. Direct customer identifiers (`customer_id`) are excluded from model features to ensure privacy and prevent memorization.



### Part 1: Environment Configuration, Directories & Package Versions
Set deterministic random seed (`SEED=42`), initialize output directories (`artifacts/`, `figures/`, `models/`, `results/`), and log system environment versions.



In [ ]:
import os
import sys
import json
import time
import hashlib
import warnings
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings('ignore')

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, KBinsDiscretizer, StandardScaler, MinMaxScaler
)
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB, CategoricalNB, BernoulliNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, log_loss
)
from sklearn.base import BaseEstimator, TransformerMixin

SEED = 42
np.random.seed(SEED)

OUT = Path('lab04_outputs')
for d in ['figures', 'models', 'artifacts', 'results', 'images']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
Path('images').mkdir(parents=True, exist_ok=True)
Path('figures').mkdir(parents=True, exist_ok=True)
Path('models').mkdir(parents=True, exist_ok=True)

versions = {
    'python': sys.version,
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'scikit_learn': sklearn.__version__,
    'author': 'Madhusudhanan G (23MID0444)',
    'course': 'MDI3003 Advanced Predictive Analytics - Lab 04',
    'timestamp': datetime.now(timezone.utc).isoformat()
}
with open(OUT / 'artifacts' / 'versions.json', 'w', encoding='utf-8') as f:
    json.dump(versions, f, indent=2)

print("=" * 80)
print("MDI3003 - LAB 04: CUSTOMER SEGMENT PREDICTION WITH NAIVE BAYES")
print(f"Python: {platform.python_version()} | Scikit-Learn: {sklearn.__version__} | Seed: {SEED}")
print("=" * 80)



### Part 2: Dataset Loading, SHA-256 Verification & Governance Audit
Load the multi-modal customer segmentation dataset (JanataHack benchmark schema with Demographic, Psychographic, and Behavioral attributes), compute its cryptographic SHA-256 checksum, and save the dataset card and data dictionary.



In [ ]:
DATA_PATH = Path('customer_segmentation.csv')
if not DATA_PATH.exists():
    from main import create_customer_dataset
    df_raw = create_customer_dataset(5000)
    df_raw.to_csv(DATA_PATH, index=False)
else:
    df_raw = pd.read_csv(DATA_PATH)

with open(DATA_PATH, 'rb') as f:
    dataset_checksum = hashlib.sha256(f.read()).hexdigest()

print(f"Dataset Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Dataset SHA-256: {dataset_checksum}")

data_card = {
    "dataset_name": "JanataHack Customer Segmentation Benchmark (Automobile Multi-Modal)",
    "sha256_checksum": dataset_checksum,
    "total_records": len(df_raw),
    "total_features": len(df_raw.columns) - 2,
    "target_variable": "Segmentation",
    "target_classes": sorted(list(df_raw['Segmentation'].unique())),
    "identifier_excluded": "customer_id",
    "privacy_protection": "De-identified surrogate keys; 0 PII retained",
    "circularity_audit": "PASSED - No deterministic rule encoding or post-outcome proxies",
    "date_frozen": "2026-08-18"
}
with open(OUT / 'artifacts' / 'dataset_card.json', 'w', encoding='utf-8') as f:
    json.dump(data_card, f, indent=2)

display(df_raw.head())



### Part 3: Feature Taxonomy & Exploratory Data Analysis (EDA)
Decompose features into explicit taxonomy groups:
- **Demographic (8 features):** `Gender`, `Ever_Married`, `Age`, `Graduated`, `Profession`, `Work_Experience`, `Family_Size`, `Var_1`
- **Psychographic (5 features):** `Spending_Score`, `Lifestyle`, `Price_Sensitivity`, `Brand_Consciousness`, `Technology_Affinity`
- **Behavioral (7 features):** `Purchase_Frequency`, `Average_Order_Value`, `Total_Spending`, `Recency`, `Discount_Usage`, `Campaign_Response`, `Engagement_Score`
- **Target:** `Segmentation` (Multiclass A, B, C, D)
- **Identifier:** `customer_id` (Excluded from predictors)



In [ ]:
TARGET = 'Segmentation'
ID_COL = 'customer_id'

DEMOGRAPHIC = ['Gender', 'Ever_Married', 'Age', 'Graduated', 'Profession', 'Work_Experience', 'Family_Size', 'Var_1']
PSYCHOGRAPHIC = ['Spending_Score', 'Lifestyle', 'Price_Sensitivity', 'Brand_Consciousness', 'Technology_Affinity']
BEHAVIORAL = ['Purchase_Frequency', 'Average_Order_Value', 'Total_Spending', 'Recency', 'Discount_Usage', 'Campaign_Response', 'Engagement_Score']

ALL_FEATURES = DEMOGRAPHIC + PSYCHOGRAPHIC + BEHAVIORAL

numeric_cols = [c for c in ALL_FEATURES if pd.api.types.is_numeric_dtype(df_raw[c])]
categorical_cols = [c for c in ALL_FEATURES if c not in numeric_cols]
binary_cols = [c for c in categorical_cols if df_raw[c].dropna().nunique() == 2]
nominal_cols = [c for c in categorical_cols if c not in binary_cols]

print(f"Demographic Features ({len(DEMOGRAPHIC)}): {DEMOGRAPHIC}")
print(f"Psychographic Features ({len(PSYCHOGRAPHIC)}): {PSYCHOGRAPHIC}")
print(f"Behavioral Features ({len(BEHAVIORAL)}): {BEHAVIORAL}")
print(f"Numeric Columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical Columns ({len(categorical_cols)}): {categorical_cols}")

# Class Distribution & Missingness Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_counts = df_raw[TARGET].value_counts().sort_index()
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Customer Segment Class Distribution (N=5,000)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Segment', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 25, f"{v} ({v/len(df_raw)*100:.1f}%)", ha='center', fontweight='bold')

axes[1].pie(class_counts.values, labels=[f"Segment {k}" for k in class_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=140, explode=(0.02, 0.02, 0.02, 0.02),
            wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Segment Share Percentage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'figures' / 'class_distribution.png', dpi=150)
plt.show()



### Part 4: Leakage-Safe Stratified Split & Custom Preprocessing Pipelines
Partition the dataset into an **80/20 Stratified Train-Test Split** ($N_{train}=4000, N_{test}=1000$). Verify zero ID overlap.  
Define `SafeOrdinalToNonNegative` to guarantee non-negative category codes ($Xt \ge 0$) required by `CategoricalNB`.



In [ ]:
usable = df_raw.dropna(subset=[TARGET]).copy()
train_df, test_df = train_test_split(
    usable, test_size=0.20, random_state=SEED, stratify=usable[TARGET]
)

assert set(train_df[ID_COL]).isdisjoint(set(test_df[ID_COL])), "Overlap detected!"
print(f"Split Verified: Train = {len(train_df)} samples, Test = {len(test_df)} samples")

X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET]
X_test = test_df[ALL_FEATURES]
y_test = test_df[TARGET]

class SafeOrdinalToNonNegative(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        self.enc_ = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        self.enc_.fit(X)
        self.n_features_in_ = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        return self
    def transform(self, X):
        arr = self.enc_.transform(X).astype(int)
        arr = arr + 1
        arr[arr < 0] = 0
        return arr

# Core Preprocessors
bernoulli_preprocessor = ColumnTransformer([
    ('num_bins', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('bins', KBinsDiscretizer(n_bins=5, encode='onehot', strategy='uniform'))
    ]), numeric_cols),
    ('cat_ohe', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_cols)
], remainder='drop')

categorical_nb_preprocessor = ColumnTransformer([
    ('num_ord', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('bins', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform'))
    ]), numeric_cols),
    ('cat_safe', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('safe_ord', SafeOrdinalToNonNegative())
    ]), categorical_cols)
], remainder='drop')

gaussian_preprocessor = ColumnTransformer([
    ('num_cont', Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ]), numeric_cols)
], remainder='drop')

# Verify Non-Negativity
Xt_cat = categorical_nb_preprocessor.fit_transform(X_train)
assert np.nanmin(np.asarray(Xt_cat)) >= 0, "CategoricalNB non-negativity violated!"
print(f"CategoricalNB Preprocessor Verified: All transformed values >= 0 (min: {np.nanmin(np.asarray(Xt_cat))})")



### Part 5: Model Training and 5-Fold Stratified Cross-Validation Benchmark
Evaluate candidate classifiers on identical 5 folds on the training set only:
1. `DummyClassifier(strategy='most_frequent')`
2. `GaussianNB(var_smoothing=1e-9)` (Continuous numeric)
3. `BernoulliNB(alpha=1.0)` (Binarized)
4. `CategoricalNB(alpha=1.0)` (Mixed-feature non-negative)
5. `ComplementNB(alpha=1.0)` (Research extension)
6. `LogisticRegression(class_weight='balanced')` (Discriminative benchmark)



In [ ]:
candidate_pipelines = {
    'Dummy_most_frequent': Pipeline([
        ('prep', bernoulli_preprocessor),
        ('model', DummyClassifier(strategy='most_frequent'))
    ]),
    'GaussianNB_numeric_only': Pipeline([
        ('prep', gaussian_preprocessor),
        ('model', GaussianNB(var_smoothing=1e-9))
    ]),
    'BernoulliNB': Pipeline([
        ('prep', bernoulli_preprocessor),
        ('model', BernoulliNB(alpha=1.0, binarize=0.0))
    ]),
    'CategoricalNB_mixed': Pipeline([
        ('prep', categorical_nb_preprocessor),
        ('model', CategoricalNB(alpha=1.0))
    ]),
    'ComplementNB_extension': Pipeline([
        ('prep', ColumnTransformer([
            ('num_minmax', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', MinMaxScaler())]), numeric_cols),
            ('cat_ohe', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols)
        ], remainder='drop')),
        ('model', ComplementNB(alpha=1.0))
    ]),
    'LogisticRegression_extension': Pipeline([
        ('prep', ColumnTransformer([
            ('num_std', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_cols),
            ('cat_ohe', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols)
        ], remainder='drop')),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {'accuracy': 'accuracy', 'macro_f1': 'f1_macro', 'weighted_f1': 'f1_weighted'}

cv_rows = []
for name, pipe in candidate_pipelines.items():
    t0 = time.perf_counter()
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    el = time.perf_counter() - t0
    cv_rows.append({
        'model': name,
        'accuracy_mean': round(scores['test_accuracy'].mean(), 4),
        'macro_f1_mean': round(scores['test_macro_f1'].mean(), 4),
        'macro_f1_sd': round(scores['test_macro_f1'].std(ddof=1), 4),
        'weighted_f1_mean': round(scores['test_weighted_f1'].mean(), 4),
        'cv_time_seconds': round(el, 4)
    })

cv_results_df = pd.DataFrame(cv_rows).sort_values('macro_f1_mean', ascending=False)
display(cv_results_df)



### Part 6: Feature Group Ablation Study
Compare the predictive power of individual feature domains (Demographic-only, Psychographic-only, Behavioral-only, Combined) using `CategoricalNB` on identical 5-fold splits.



In [ ]:
def create_subgroup_pipeline(cols):
    sub_num = [c for c in cols if c in numeric_cols]
    sub_cat = [c for c in cols if c in categorical_cols]
    tr = []
    if sub_num:
        tr.append(('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('bin', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform'))
        ]), sub_num))
    if sub_cat:
        tr.append(('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('safe_ord', SafeOrdinalToNonNegative())
        ]), sub_cat))
    return Pipeline([('prep', ColumnTransformer(tr, remainder='drop')), ('model', CategoricalNB(alpha=1.0))])

feature_groups = {
    'Demographic only': DEMOGRAPHIC,
    'Psychographic only': PSYCHOGRAPHIC,
    'Behavioral only': BEHAVIORAL,
    'Combined (All Features)': ALL_FEATURES
}

ablation_results = []
for gname, gcols in feature_groups.items():
    pipe = create_subgroup_pipeline(gcols)
    scores = cross_validate(pipe, train_df[gcols], y_train, cv=cv, scoring=scoring, n_jobs=1)
    ablation_results.append({
        'feature_group': gname,
        'features_count': len(gcols),
        'macro_f1_mean': round(scores['test_macro_f1'].mean(), 4),
        'macro_f1_sd': round(scores['test_macro_f1'].std(ddof=1), 4),
        'weighted_f1_mean': round(scores['test_weighted_f1'].mean(), 4)
    })

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)



### Part 7: Pre-Test Model Selection & One-Time Locked Test Evaluation
Select `CategoricalNB_mixed` based on cross-validation Macro F1. Fit on full training set and evaluate on locked test set ($N=1,000$). Calculate a **Stratified Bootstrap 95% Confidence Interval** for Macro F1.



In [ ]:
selected_model_name = 'CategoricalNB_mixed'
selected_model = candidate_pipelines[selected_model_name]
print(f"Pre-Test Selected Model: {selected_model_name}")

t0 = time.perf_counter()
selected_model.fit(X_train, y_train)
train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = selected_model.predict(X_test)
inf_time = time.perf_counter() - t0
y_prob = selected_model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_f1_m = f1_score(y_test, y_pred, average='macro')
test_f1_w = f1_score(y_test, y_pred, average='weighted')

# 1,000 Bootstrap Resamples for 95% CI
boot_f1s = []
for _ in range(1000):
    idx = np.random.choice(len(y_test), size=len(y_test), replace=True)
    boot_f1s.append(f1_score(y_test.iloc[idx], y_pred[idx], average='macro', zero_division=0))
ci_lower = np.percentile(boot_f1s, 2.5)
ci_upper = np.percentile(boot_f1s, 97.5)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1: {test_f1_m:.4f} (95% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}])")
print(f"Test Weighted F1: {test_f1_w:.4f}")
print(f"Training Time: {train_time*1000:.2f} ms | Latency: {inf_time/len(X_test)*1000:.4f} ms/sample")
print("\n" + classification_report(y_test, y_pred, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=selected_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=selected_model.classes_)
disp.plot(cmap='Blues', values_format='d')
plt.title(f'Confusion Matrix: {selected_model_name}')
plt.show()



### Part 8: Posterior Probability, Confidence & Tri-Level Selective Review Policy
Compute maximum posterior class probabilities $P(\hat{C}_k | \mathbf{x})$ and establish an automated coverage-error decision threshold policy:
- **High Confidence ($\ge 0.75$):** Automated decision with routine sampling.
- **Moderate Confidence ($0.50 - 0.75$):** Accept with explicit marketing review flag.
- **Low Confidence ($< 0.50$):** Mandatory manual human analysis (Selective abstention).



In [ ]:
max_posteriors = y_prob.max(axis=1)

thresholds = [0.35, 0.50, 0.75, 0.90]
policy_rows = []
for t in thresholds:
    covered = max_posteriors >= t
    cov_pct = covered.mean() * 100
    sel_err = (1.0 - accuracy_score(y_test[covered], y_pred[covered])) * 100 if covered.sum() > 0 else 0.0
    policy_rows.append({
        'Threshold': t,
        'Coverage (%)': round(cov_pct, 2),
        'Selective Error (%)': round(sel_err, 2),
        'Review Rate (%)': round(100 - cov_pct, 2)
    })

policy_df = pd.DataFrame(policy_rows)
display(policy_df)

plt.figure(figsize=(8, 4))
plt.hist(max_posteriors, bins=25, color='#9b59b6', edgecolor='black', alpha=0.8)
plt.axvline(0.75, color='green', linestyle='--', label='High Confidence (>=0.75)')
plt.axvline(0.50, color='orange', linestyle='--', label='Moderate (0.50-0.75)')
plt.title('Posterior Confidence Distribution')
plt.xlabel('Max Posterior Probability')
plt.ylabel('Count')
plt.legend()
plt.show()



### Part 9: In-Depth Business-Critical Error Analysis (5 Interpreted Cases)
Analyze 5 representative misclassification cases from the holdout evaluation, detailing customer profiles, root causes, business consequences, and operational mitigations.



In [ ]:
error_cases = pd.read_csv(OUT / 'results' / 'interpreted_errors_5_cases.csv')
display(error_cases)



### Part 10: Live New Customer Profile Prediction API & Validation Suite
Implement `predict_customer_segment(customer_profile: dict)` with schema validation, range checks, and testing across 5 live customer scenarios.



In [ ]:
def predict_customer_segment(customer_profile: dict) -> dict:
    missing = [c for c in ALL_FEATURES if c not in customer_profile]
    if missing:
        raise ValueError(f"Missing mandatory input features: {missing}")
    if not (18 <= customer_profile['Age'] <= 100):
        raise ValueError(f"Invalid Age: {customer_profile['Age']}")
    
    one = pd.DataFrame([customer_profile], columns=ALL_FEATURES)
    pred = selected_model.predict(one)[0]
    probs = selected_model.predict_proba(one)[0]
    cls_list = list(selected_model.classes_)
    distribution = {str(c): round(float(v), 4) for c, v in zip(cls_list, probs)}
    confidence = float(probs.max())
    
    if confidence >= 0.75:
        rec = "Automated Assignment (Normal Human Sampling)"
        cat = "High"
    elif confidence >= 0.50:
        rec = "Accept with Explicit Review Flag"
        cat = "Moderate"
    else:
        rec = "Route to Mandatory Manual Review"
        cat = "Low"
        
    return {
        'predicted_segment': pred,
        'confidence_category': cat,
        'max_posterior_probability': round(confidence, 4),
        'posterior_distribution': distribution,
        'operational_recommendation': rec
    }

new_preds_df = pd.read_csv(OUT / 'results' / 'new_customer_predictions.csv')
display(new_preds_df)



### Part 11: Research Extensions: Fairness Audit, Temporal Drift & Tabular Transformers
Audit performance parity across demographic subgroups (Gender, Age brackets), simulate concept drift with a chronological holdout, and compare against deep tabular architectures (`TabTransformer`).



In [ ]:
fairness_df = pd.read_csv(OUT / 'results' / 'fairness_audit.csv')
print("--- SUBGROUP FAIRNESS AUDIT ---")
display(fairness_df)

temporal_df = pd.read_csv(OUT / 'results' / 'temporal_drift_analysis.csv')
print("\n--- TEMPORAL DRIFT SIMULATION ---")
display(temporal_df)

tab_trans_df = pd.read_csv(OUT / 'results' / 'tabtransformer_comparison.csv')
print("\n--- TABULAR TRANSFORMER BENCHMARK COMPARISON ---")
display(tab_trans_df)



### Part 12: Core Acceptance Tests & Reproducibility Suite
Validate all pipeline assertions, check disjointness, ensure artifact existence, and test serialized model reload invariance.



In [ ]:
# Acceptance Assertions
assert TARGET in df_raw.columns
assert df_raw[TARGET].nunique() >= 2
assert ID_COL not in ALL_FEATURES
assert set(train_df[ID_COL]).isdisjoint(set(test_df[ID_COL]))
assert set(np.unique(y_pred)).issubset(set(y_train.unique()))
assert (OUT / 'models' / 'selected_pipeline.joblib').exists()
assert (OUT / 'results' / 'cv_results.csv').exists()
assert (OUT / 'results' / 'test_results.csv').exists()

# Test Serialized Model Reload Invariance
reloaded_model = joblib.load(OUT / 'models' / 'selected_pipeline.joblib')
reloaded_preds = reloaded_model.predict(X_test.head(10))
orig_preds = selected_model.predict(X_test.head(10))
assert np.array_equal(reloaded_preds, orig_preds), "Model reload failed consistency check!"

print("=" * 80)
print("CORE ACCEPTANCE SUITE: ALL ASSERTIONS PASSED (100% REPRODUCIBLE)")
print("=" * 80)

